# SpatialData tiling POC with Celldega

Load the modified SpatialData store, check its new storage features, and visualize it directly with Celldega.

In [ ]:
from pathlib import Path
import warnings
import spatialdata as sd

warnings.filterwarnings("ignore", message="The table is annotating 'cell_labels'")
store = Path("data/skin_adapt_v2.zarr")
sdata = sd.read_zarr(store)
sdata

In [ ]:
import pyarrow.parquet as pq
from scipy.sparse import isspmatrix_csc

point_schema = pq.read_schema(next((store / "points/transcripts/points.parquet").glob("*.parquet")))
shape_schema = pq.read_schema(next((store / "shapes/cell_boundaries/shapes.parquet").glob("*.parquet")))

print("transcripts: NGFF points using x/y columns =", {"x", "y"} <= set(point_schema.names))
print("cell boundaries:", shape_schema.field("geometry").metadata[b"ARROW:extension:name"].decode())
print("cell-by-gene X_csc layer:", isspmatrix_csc(sdata.tables["table"].layers["X_csc"]))

In [ ]:
import celldega as dega

port = dega.viz.get_local_server()
dega.viz.Landscape(
    base_url=f"http://127.0.0.1:{port}/{store.as_posix()}",
    technology="Xenium",
    dataset_name="skin_adapt_v2",
    transform=[[4.705882352941177, 0, 0], [0, 4.705882352941177, 0], [0, 0, 1]],
    ini_x=21_375,
    ini_y=10_250,
    ini_zoom=-5,
)